# EDA, datos y preprocesamiento

Este cuaderno reproduce el análisis exploratorio del dashboard con las mismas funciones (`dashboard/datos.py` y `dashboard/figuras.py`). Todas las figuras son interactivas.

In [1]:
import os  # Variables de entorno
import sys  # Configuración del intérprete
os.environ.setdefault("LOKY_MAX_CPU_COUNT", str(os.cpu_count()))  # Evita un aviso de joblib en Windows
sys.path.insert(0, "..")  # Agrega la raíz del proyecto para importar src y dashboard
import warnings  # Control de avisos
warnings.filterwarnings("ignore")  # Oculta avisos de versiones que no afectan los resultados
import numpy as np  # Cálculo numérico
import pandas as pd  # Tablas
from IPython.display import HTML, display  # Incrustar figuras interactivas
from dashboard import figuras as fg  # Mismas figuras del dashboard
from src import config as cf  # Configuración del proyecto
pd.set_option("display.width", 160)  # Ancho de las tablas
pd.set_option("display.max_columns", 30)  # Columnas visibles


def mostrar(fig):  # Muestra una figura del dashboard como HTML interactivo
    """Aplica el fondo oscuro de las tarjetas y la incrusta con Plotly cargado desde internet."""
    fig.update_layout(paper_bgcolor=fg.SUPERFICIE, plot_bgcolor=fg.SUPERFICIE)  # Fondo oscuro
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False)))  # Figura interactiva
from dashboard import datos as dd  # Panel, agregados del EDA y artefactos
print(f"Panel completo: {dd.RAW.shape[0]:,} filas · {dd.RAW['COD_MUN_N'].nunique()} municipios · {dd.RAW['time_idx'].nunique()} semanas")  # Dimensión
print(f"Filas para modelar (2015–2024): {len(dd.MOD):,}")  # Filas útiles

Panel completo: 294,032 filas · 376 municipios · 782 semanas
Filas para modelar (2015–2024): 191,384


## Fuente, estructura y estadística descriptiva

Panel balanceado municipio × semana (2010–2024) que integra la notificación semanal de dengue con clima, NDVI, el índice ONI, población y flujo de personas.

In [2]:
cols = ["casos_totales", "temp_mean", "prec_total", "ndvi_mean", "oni_anom", "Flujo_in", "poblacion"]  # Variables originales
desc = dd.RAW[cols].describe().T  # Media, desviación, cuartiles, mínimo y máximo
desc["asimetria"] = dd.RAW[cols].skew()  # Asimetría
desc.round(2)  # Tabla

,count,mean,std,min,25%,50%,75%,max,asimetria
casos_totales,294032.0,4.29,24.72,0.00,0.00,0.00,2.00,1426.00,21.22
temp_mean,294032.0,23.20,3.03,9.80,22.23,22.46,24.30,34.07,-0.19
prec_total,294032.0,42.78,52.00,0.00,4.30,25.70,62.00,1053.00,2.30
ndvi_mean,294032.0,0.63,0.11,0.09,0.57,0.65,0.72,0.88,-0.80
oni_anom,294032.0,0.05,0.90,-1.64,-0.60,-0.14,0.57,2.75,0.82
Flujo_in,294032.0,3.62,21.29,0.00,0.28,0.67,1.77,1337.56,24.55
poblacion,294032.0,85757.93,410379.23,3179.00,12156.25,22490.00,43434.75,7149540.00,14.31


**Interpretación.** Los casos semanales son muy asimétricos (mediana 0, máximo de más de 1 400): justifican el logaritmo. El flujo de personas y la población también tienen colas largas.

## Casos por año y serie nacional

In [3]:
mostrar(fg.fig_anual(dd.ANUAL))  # Casos por año con el periodo del modelo
mostrar(fg.fig_serie_nacional(dd.NAC, dd.CORTES))  # Casos semanales y El Niño

**Interpretación.** Las epidemias de 2010, 2013, 2019 y 2023–2024 coinciden con episodios de El Niño (ONI > 0,5). 2024 es un régimen excepcional: la prueba evalúa a los modelos en condiciones más extremas que las del entrenamiento.

## Estacionalidad y grupos de población

In [4]:
mostrar(fg.fig_estacionalidad(dd.EST))  # Perfil semanal de casos y lluvia
mostrar(fg.fig_grupos(dd.EDAD, dd.SEXO))  # Casos por edad y sexo

## Canal endémico: definición del brote

Una semana es de brote si sus casos superan el tercer cuartil de las mismas semanas (± 2) de los cinco años anteriores y hay al menos 3 casos.

In [5]:
for mun in [76001, 68001]:  # Cali y Bucaramanga
    s = dd.RAW[(dd.RAW["COD_MUN_N"] == mun) & (dd.RAW["ANO"] >= 2015)]  # Serie del municipio
    mostrar(fg.fig_canal(pd.DataFrame({"fecha": s["week_start"], "casos": s["casos_totales"], "umbral": s["umbral"], "brote": s["brote"]}), dd.nombre_municipio(mun)))  # Figura
print(f"Semanas de brote 2015–2024: {int(dd.RAW['brote'].sum()):,} ({dd.RAW['brote'].mean():.1%})")  # Prevalencia

Semanas de brote 2015–2024: 28,081 (14.4%)


## Clima de hoy y brote en cuatro semanas

In [6]:
for v in ["temp_mean_m8", "prec_total_m8", "oni_l8"]:  # Tres variables climáticas
    mostrar(fg.fig_tasa_por_decil(dd.deciles(v), v))  # Tasa de brote por decil
mostrar(fg.fig_ccf(dd.CCF))  # Correlación cruzada clima → casos
mostrar(fg.fig_oni_anual(dd.ONI_ANUAL))  # El Niño y los años epidémicos

**Interpretación.** La tasa de brote crece en los deciles de temperatura de 25–28 °C, la zona óptima para *Aedes aegypti*. La correlación cruzada es débil (|r| ≈ 0,05) y llega con 8–12 semanas de rezago: el clima modula el riesgo, pero no lo determina solo.

## Departamentos, faltantes y atípicos

In [7]:
mostrar(fg.fig_departamentos(dd.DEP))  # Incidencia por departamento
mostrar(fg.fig_faltantes(dd.FALTANTES))  # Temperatura imputada por año
print(f"Temperatura imputada: {dd.RAW['temp_mean_missing'].mean():.1%} de las semanas")  # Faltantes
from dashboard.sec_eda import tabla_atipicos  # Regla de Tukey sobre las variables del modelo
mostrar(fg.fig_atipicos(tabla_atipicos()))  # Tasa de brote en atípicos

Temperatura imputada: 64.3% de las semanas


**Interpretación.** No hay nulos, pero la temperatura está imputada en casi dos tercios de las semanas (concentrada cerca de 22 °C), lo que atenúa la señal térmica; se conserva la bandera `temp_missing`. Los atípicos son epidemias reales y no se eliminan.

## Relaciones entre variables

In [8]:
mostrar(fg.fig_correlacion_objetivo(dd.CORR_Y))  # Correlación con el brote en t + 4
mostrar(fg.fig_matriz_correlacion(dd.CORR))  # Matriz de correlación

## Prevención de la fuga de datos

* Todas las variables usan semanas $\le t$; el umbral objetivo solo usa años anteriores.
* La normal climática (anomalías) se calcula con 2010–2021.
* La validación respeta el tiempo (hacia adelante por años).
* El preprocesamiento vive dentro del `Pipeline`.

La demostración siguiente compara la partición correcta con dos fugas típicas.

In [9]:
if dd.ART:  # Requiere el entrenamiento
    mostrar(fg.fig_fuga(dd.ART["fuga"]))  # Fugas: partición aleatoria y variable del futuro
    display(pd.DataFrame(dd.ART["fuga"]).drop(columns="azar").T.round(4))  # Tabla

,auc_roc,auc_pr
correcto,0.8975,0.8221
futuro,1.0000,0.9999
aleatoria,0.9358,0.7721
